In [ ]:
# This is an example GPT style decoder only transformer model and example dataset
# This an example of the use of the icanswim/cosmosis repo for data science and 
# machine learning projects

In [1]:
import sys # required for relative imports in jupyter lab
sys.path.insert(0, '../')

from torch import long
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from cosmosis.dataset import AsTensor
from cosmosis.learning import Learn, Selector, Metric
from cosmosis.model import GPT

from dataset import TinyShakes

In [7]:
# explore the ds

ds_param = {'transforms': {'tokens': [AsTensor(long)],
                           'y': [AsTensor(long)],
                           'position': [AsTensor(long)]},
            'd_seq': 6,
            'n': 5,
            'prompt': False,
            'dir': './data'}

ts = TinyShakes(**ds_param)

print(ts[0])
print(ts[0]['tokens'].shape, ts[0]['tokens'].dtype)
print('decoded tokens: ', ts.encoding.decode(ts[0]['tokens'].tolist()))
print('decoded y: ', ts.encoding.decode(ts[0]['y'].tolist()))


{'tokens': tensor([ 5962, 22307,    25,  7413,   356,  5120]), 'y': tensor([22307,    25,  7413,   356,  5120,   597]), 'position': tensor([0, 1, 2, 3, 4, 5])}
torch.Size([6]) torch.int64
decoded tokens:  First Citizen: Before we proceed
decoded y:   Citizen: Before we proceed any


In [14]:
print(len(ts.ds_idx))
print(len(ts.ds))

301324
301332


In [4]:
# example using prompt for inference

ds_param = {'transforms': {'tokens': [AsTensor(long)],
                           'y': [AsTensor(long)],
                           'position': [AsTensor(long)]},
            'prompt':'First Citizen: Before we proceed any further, hear '}

prompt = TinyShakes(**ds_param)
print(prompt[0])
print(prompt[0]['tokens'])
print(prompt[0]['y']) 
print('decoded tokens: ', prompt.encoding.decode(prompt[0]['tokens'].tolist()))
print('decoded y: ', prompt.encoding.decode(prompt[0]['y'].tolist()))

{'tokens': tensor([ 5962, 22307,    25,  7413,   356,  5120,   597,  2252,    11,  3285,
          220]), 'y': tensor([22307,    25,  7413,   356,  5120,   597,  2252,    11,  3285,   220]), 'position': tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])}
tensor([ 5962, 22307,    25,  7413,   356,  5120,   597,  2252,    11,  3285,
          220])
tensor([22307,    25,  7413,   356,  5120,   597,  2252,    11,  3285,   220])
decoded tokens:  First Citizen: Before we proceed any further, hear 
decoded y:   Citizen: Before we proceed any further, hear 


In [26]:
# pass a single example from dataset to model to loss function
# (batch, d_seq, d_model)

d_seq = 3 # dimension sequence
d_vocab = 50304 # dimension vocabulary
d_vec = 4 # dimension embedding vector
d_model = 4 # dimension model input

assert d_model == d_vec

ds_param = {'transforms': {'tokens': [AsTensor(long)],
                           'y': [AsTensor(long)],
                           'position': [AsTensor(long)]},
            'd_seq': d_seq,
            'n': 5,
            'prompt': False}

ts = TinyShakes(**ds_param)

print(ts[0])
print(ts[0]['tokens'].shape, ts[0]['tokens'].dtype)
print(ts[0]['y'].shape, ts[0]['y'].dtype)
print('decoded tokens: ', ts.encoding.decode(ts[0]['tokens'].tolist()))
print('decoded y: ', ts.encoding.decode(ts[0]['y'].tolist()))

model_param = {'device': 'cpu',
               'd_model': d_model, # matches embedding dimension
               'd_vocab': d_vocab, 
               'n_head': 2, 
               'num_layers': 2,
               'd_vec': d_vec,
               'embed_param': {'tokens': (d_vocab, d_vec, None, True), 
                               'y': (d_vocab, d_vec, None, True),
                               'position': (d_seq, d_vec, None, True)}} 

gpt = GPT(model_param)

data = ts[0]
out = gpt(data)
print('output: ', out, out.shape, out.dtype)

prompt_tokens = data['tokens']
print('prompt_tokens: ', prompt_tokens, prompt_tokens.shape, prompt_tokens.dtype)

target_tokens = data['y']
print('target_tokens: ', target_tokens, target_tokens.shape, target_tokens.dtype)

generated_embeddings = out.squeeze()
print('generated_embeddings: ', generated_embeddings, generated_embeddings.shape, generated_embeddings.dtype)
print('decoded generated tokens: ', prompt.encoding.decode(generated_embeddings.argmax(dim=-1).tolist()))

cel_func = CrossEntropyLoss()
loss = cel_func(out, target_tokens)
print('loss: ', loss)


{'tokens': tensor([ 5962, 22307,    25]), 'y': tensor([22307,    25,  7413]), 'position': tensor([0, 1, 2])}
torch.Size([3]) torch.int64
torch.Size([3]) torch.int64
decoded tokens:  First Citizen:
decoded y:   Citizen: Before
output:  tensor([[-0.0387, -0.0369,  0.0419,  ...,  0.0241, -0.0077,  0.0609],
        [ 0.0562,  0.0347, -0.0194,  ..., -0.0234,  0.0499, -0.0750],
        [ 0.0819,  0.0421, -0.0169,  ..., -0.0437,  0.0279,  0.0298]],
       grad_fn=<MmBackward0>) torch.Size([3, 50304]) torch.float32
prompt_tokens:  tensor([ 5962, 22307,    25]) torch.Size([3]) torch.int64
target_tokens:  tensor([22307,    25,  7413]) torch.Size([3]) torch.int64
generated_embeddings:  tensor([[-0.0387, -0.0369,  0.0419,  ...,  0.0241, -0.0077,  0.0609],
        [ 0.0562,  0.0347, -0.0194,  ..., -0.0234,  0.0499, -0.0750],
        [ 0.0819,  0.0421, -0.0169,  ..., -0.0437,  0.0279,  0.0298]],
       grad_fn=<SqueezeBackward0>) torch.Size([3, 50304]) torch.float32
decoded generated tokens:  oes366

In [23]:
# put all together in a learner
# (batch, d_seq, d_model)

d_seq = 100 # dimension sequence
d_vocab = 50304 # dimension vocabulary
d_vec = 768 # dimension embedding vector
d_model = 768 # dimension model input
assert d_model == d_vec

ds_param = {'train_param': {'transforms': {'tokens': [AsTensor(long)],
                                           'y': [AsTensor(long)],
                                           'position': [AsTensor(long)]},
                            'd_seq': d_seq,
                            #'n': 1000,
                            'prompt': False,
                           }}

model_param = {'d_model': d_model,
               'd_vocab': d_vocab, 
               'n_head': 12, 
               'num_layers': 12,
               'd_seq': d_seq,
               'd_vec': d_vec,
               'embed_param': {'tokens': (d_vocab, d_vec, None, True), 
                               'y': (d_vocab, d_vec, None, True),
                               'position': (d_seq, d_vec, None, True)}} 
                                       
metric_param = {'metric_name': 'transformer',
                 'report_interval': 1,
                 'min_lr': .0025} # break if learning rate falls below                        
             
opt_param = {'lr': 0.01}

crit_param = {}

sample_param = {'set_seed': 88,
                'splits': (.7,.15)}

sched_param = {'factor': .5, 
               'patience': 2,
               'cooldown': 2}

learn = Learn([TinyShakes], 
              GPT,
              Metric=Metric,
              Sampler=Selector, 
              Optimizer=Adam, 
              Scheduler=ReduceLROnPlateau, 
              Criterion=CrossEntropyLoss,
              model_param=model_param, ds_param=ds_param, sample_param=sample_param,
              opt_param=opt_param, sched_param=sched_param, crit_param=crit_param,
              metric_param=metric_param, 
              batch_size=32, epoch=2, gpu=False, save_model='tinyshakes768', 
              load_model=None, target='y')

In [6]:
# inference
d_gen = 50 # dimension generate number of tokens
d_vocab = 50304 # dimension vocabulary
d_vec = 768 # dimension embedding vector
d_model = 768 # dimension model input
d_pos = 50 # dimension positional encoding d_pos >= max(len(prompt_tokens), d_gen)

assert d_model == d_vec


ds_param = {'train_param': {'transforms': {'tokens': [AsTensor()],
                                           'y': [AsTensor()],
                                           'position': [AsTensor()]},
                            'prompt': 'Farewell! God knows when we shall meet'}
           }

model_param = {
               'd_model': d_model,
               'd_vocab': d_vocab, 
               'n_head': 12, 
               'num_layers': 12,
               'd_gen': d_gen,
               'd_vec': d_vec,
               'temperature': 1000,
               'top_k': 3,
               'embed_param': {'tokens': (d_vocab, d_vec, None, True), 
                               #'y': (d_vocab, d_vec, None, True),
                               'position': (d_pos, d_vec, None, True)},
              } 
                                       
metrics_param = {'metric_name': 'transformer'}                        
             
opt_param = {}

crit_param = {}

sample_param = {}

sched_param = {}

learn = Learn([TinyShakes], 
              GPT,
              Metrics=Metrics,
              Sampler=Selector, 
              Optimizer=None, 
              Scheduler=None, 
              Criterion=None, # no criterion implies inference
              model_param=model_param, ds_param=ds_param, sample_param=sample_param,
              opt_param=opt_param, sched_param=sched_param, crit_param=crit_param,
              metrics_param=metrics_param, 
              batch_size=1, epochs=5, gpu=True, 
              load_model='tinyshakes768.pth', load_embed=True, target=None)


len(self.ds_idx):  1
data.nbytes:  20
CDataset created...
applying _init_weights...
GPT model loaded...
number of model parameters:  38634240
model loaded from state_dict...
loading embedding weights...
running model on gpu...

.....................

total learning time: 0:00:01.169133
predictions:  [["IDll, , not he march not again, and, sir, and, and the air And so.  KING RICHARD III: I will not, and, And with the city.  DUKE OF YORK: I'll warrant"]]
inference instance 2025-08-15 07:55:22.136647 complete and saved to csv...

.....................

total learning time: 0:00:01.398799
predictions:  [["ID!, 's not he march be again-morrow Shall falconceived in the king.  DUKE: O, And so much.  KING RICHARD III: What is a bawdello, And therefore, and,"]]
inference instance 2025-08-15 07:55:22.366313 complete and saved to csv...

.....................

total learning time: 0:00:01.620027
predictions:  [['IDll,  save not he march be together, and, I amain, and, That thou hast shown it.  KI